# RHNA & Housing Production

This notebook downloads, cleans, validates, and exports RHNA (Regional Housing
Needs Allocation) and housing-production data for the 18 incorporated
jurisdictions in San Diego County and for San Diego County as a whole.

**This pass loads and reviews all six source datasets for this workstream:**

1. **RHNA** -- 6th Cycle targets by income category (HCD/CKAN)
2. **APR** -- building permits and completions by income category (HCD/CKAN, Table A2)
3. **City of SD permits** -- local permit-level detail (data.sandiego.gov, Socrata)
4. **DOF** -- population and housing-unit estimates (CA Dept. of Finance, manual download)
5. **Census / ACS** -- population, housing units, and tenure context (Census API)

Later passes in this notebook (below) also cover: field verification, an
APR-vs-permits double-counting check, available production/stock categories,
a combined jurisdiction-year table, and documented gaps/limitations.

Source: California HCD's Housing Element Annual Progress Report (APR) open data,
published on data.ca.gov. Data is self-reported by jurisdictions to HCD and is
not independently verified by HCD.


## Setup

In [ ]:
%pip install pandas numpy requests

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def find_workstream_root() -> Path:
    """
    Find the folder containing both data/ and notebooks/.
    Works whether the notebook starts from:
    - the repository root,
    - the "jake's work" folder, or
    - the notebooks folder.
    """
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

        my_folder = candidate / "jake's work"
        if (my_folder / "data").exists() and (my_folder / "notebooks").exists():
            return my_folder

    raise FileNotFoundError(
        "Could not locate the workstream root (a folder with both "
        "data/ and notebooks/ inside it). Run this notebook from within "
        "the repository."
    )


ROOT = find_workstream_root()
RAW_DIR = ROOT / "data" / "raw" / "hcd"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"

for d in (RAW_DIR, PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Workstream root:", ROOT)


## Target data year

Per Lauren's note: use the most recent **completed** year, and keep it as
consistent as possible across sources. As of this notebook's last check
(August 2026):

| Source | Newest available | Notes |
|---|---|---|
| HCD APR/RHNA | **2025** | 2025 filings were due April 1, 2026; HCD's dataset was last refreshed May 22, 2026 |
| CA DOF population/housing | **Jan 1, 2025** (revised) | Jan 1, 2026 exists but is only *provisional* -- don't use it yet |
| ACS 5-year estimates | **2020-2024** (labeled "2024") | The 2021-2025 vintage won't release until ~Dec 2026/Jan 2027 -- 2025 ACS data does not exist yet |
| City of SD permits | rolling/current | no lag |

**Decision:** target **2025** for APR/RHNA, DOF, and City of SD permits.
ACS is the one source that structurally cannot keep up -- use its newest
vintage (2020-2024, i.e. "2024" data) and document that exception rather
than forcing a mismatch. This is still a full year newer than the ACS 2023
data currently baked into the dashboard prototype.


In [ ]:
TARGET_YEAR = 2025          # HCD APR/RHNA, DOF, City of SD permits
ACS_VINTAGE_LABEL = "2020-2024"   # newest available ACS 5-year window
ACS_DATA_YEAR = 2024        # the single "current" year that vintage represents

print(f"Targeting {TARGET_YEAR} for APR/RHNA/DOF/permits; "
      f"ACS {ACS_VINTAGE_LABEL} ({ACS_DATA_YEAR} equivalent) as the documented exception.")


In [ ]:
SAN_DIEGO_CITIES = [
    "Carlsbad", "Chula Vista", "Coronado", "Del Mar", "El Cajon",
    "Encinitas", "Escondido", "Imperial Beach", "La Mesa", "Lemon Grove",
    "National City", "Oceanside", "Poway", "San Diego", "San Marcos",
    "Santee", "Solana Beach", "Vista",
]

# County of San Diego appears in HCD data as its own jurisdiction (covers
# unincorporated areas only) -- keep it separate rather than merging it
# into the incorporated-city list above.
COUNTY_JURISDICTION_NAME = "San Diego County"

RHNA_CYCLE = "6th Cycle"
RHNA_CYCLE_YEARS = (2021, 2029)


## Download raw APR data from HCD's open data portal

HCD publishes APR data on data.ca.gov as a CKAN dataset. Rather than
hard-coding a resource ID (which can change), we look up the current
resource by name within the package -- this keeps the notebook working
even if HCD re-publishes the file under a new ID.

**Table A2** is the relevant table for this workstream: it reports building
permits issued and certificates of occupancy, by jurisdiction, year, and
income category -- exactly what feeds the "permitted units" / "completed
units" / "affordable share" metrics on the dashboard.


In [ ]:
CKAN_PACKAGE_URL = (
    "https://data.ca.gov/api/3/action/package_show"
    "?id=housing-element-annual-progress-report-apr-data-by-jurisdiction-and-year"
)


def get_json(url: str, params: dict | None = None, timeout: int = 60):
    response = requests.get(
        url,
        params=params,
        timeout=timeout,
        headers={"User-Agent": "CHPD Housing Dashboard Data Validation (housing research notebook)"},
    )
    if not response.ok:
        raise RuntimeError(
            f"Request failed with status {response.status_code}\n"
            f"URL: {response.url}\nResponse: {response.text[:1000]}"
        )
    return response.json()


def find_resource_download_url(package_json: dict, name_contains: str) -> str:
    resources = package_json["result"]["resources"]
    matches = [
        r for r in resources
        if name_contains.lower() in r.get("name", "").lower()
        and r.get("format", "").upper() == "CSV"
    ]
    if not matches:
        available = [r.get("name") for r in resources]
        raise ValueError(
            f"No CSV resource matching '{name_contains}' found. "
            f"Available resources: {available}"
        )
    # Prefer the most recently modified match if there are several.
    matches.sort(key=lambda r: r.get("last_modified", ""), reverse=True)
    return matches[0]["url"], matches[0]["name"]


package_json = get_json(CKAN_PACKAGE_URL)
table_a2_url, table_a2_name = find_resource_download_url(package_json, "Table A2")
print("Using resource:", table_a2_name)
print("Download URL:", table_a2_url)


In [ ]:
raw_path = RAW_DIR / "apr_table_a2_raw.csv"

if not raw_path.exists():
    print("Downloading Table A2 (this file is large, may take a minute)...")
    resp = requests.get(table_a2_url, timeout=300)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)
    print("Saved to", raw_path)
else:
    print("Raw file already downloaded at", raw_path)

apr_raw = pd.read_csv(raw_path, low_memory=False)
print(apr_raw.shape)
apr_raw.head()


## Filter to San Diego County jurisdictions

The raw file is statewide (500+ jurisdictions). Filter down to the 18
incorporated San Diego cities plus the County (unincorporated areas).

In [ ]:
def normalize_jurisdiction(name: object) -> str:
    """Match the cleaning convention used elsewhere in the dashboard repo."""
    s = str(name).strip().lower()
    if s == "national city":
        # Special case: "National City" is the city's actual proper name,
        # not a generic "<name> City" pattern -- the blind trailing-"city"
        # strip below would otherwise mangle it into just "national".
        # (Note: app.js in housing-dashboard-prototype has this same bug --
        # worth flagging separately, not fixing there.)
        return "national city"
    s = re.sub(r"^city\s+of\s+", "", s)
    s = re.sub(r"^county\s+of\s+", "county ", s)
    s = re.sub(r"\s+city$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


# TODO: confirm the actual jurisdiction column name once the raw file is
# downloaded -- HCD's Table A2 has historically used "Jurisdiction Name" or
# similar; print(apr_raw.columns.tolist()) to check.
JURISDICTION_COL = "Jurisdiction Name"  # <-- verify / update this

apr_raw["jur_clean"] = apr_raw[JURISDICTION_COL].map(normalize_jurisdiction)

sd_jur_keys = {normalize_jurisdiction(c) for c in SAN_DIEGO_CITIES}
sd_jur_keys.add(normalize_jurisdiction(COUNTY_JURISDICTION_NAME))

sd_apr = apr_raw[apr_raw["jur_clean"].isin(sd_jur_keys)].copy()
print("Rows for San Diego County jurisdictions:", len(sd_apr))
print("Jurisdictions found:", sorted(sd_apr["jur_clean"].unique()))


In [ ]:
# TODO: confirm the actual year column name once the raw file is downloaded
# (commonly "Reporting Year" or similar in HCD's Table A2).
YEAR_COL = "Reporting Year"  # <-- verify / update this

sd_apr[YEAR_COL] = pd.to_numeric(sd_apr[YEAR_COL], errors="coerce")

available_years = sorted(sd_apr[YEAR_COL].dropna().unique())
print("Years available in this pull:", available_years)

if TARGET_YEAR not in available_years:
    fallback = max(available_years) if available_years else "unknown"
    print(
        f"WARNING: {TARGET_YEAR} not found in the data yet. "
        f"Most recent available year is {fallback}. "
        "Some jurisdictions may not have filed their 2025 APR yet -- check "
        "which cities are missing before deciding whether to fall back to 2024."
    )

sd_apr_target_year = sd_apr[sd_apr[YEAR_COL] == TARGET_YEAR].copy()
print(f"Rows for {TARGET_YEAR}:", len(sd_apr_target_year))
missing_jurs = sd_jur_keys - set(sd_apr_target_year["jur_clean"].unique())
if missing_jurs:
    print(f"Jurisdictions with no {TARGET_YEAR} row yet: {sorted(missing_jurs)}")


## Compute production metrics

Build per-jurisdiction, per-year totals: permitted units, completed units,
and affordable share, matching the metric definitions already used in the
dashboard prototype.

In [ ]:
# TODO: update these column-name patterns once you've confirmed the real
# column names in sd_apr.columns -- HCD's raw column names are long and
# have changed slightly between publication years.
BP_INCOME_COLS = [c for c in sd_apr_target_year.columns if c.upper().startswith("BP_") and "INCOME" in c.upper()]
CO_INCOME_COLS = [c for c in sd_apr_target_year.columns if c.upper().startswith("CO_") and "INCOME" in c.upper()]

print("Building permit income columns found:", BP_INCOME_COLS)
print("Certificate of occupancy income columns found:", CO_INCOME_COLS)

sd_apr_target_year["bp_units_total"] = sd_apr_target_year[BP_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["co_units_total"] = sd_apr_target_year[CO_INCOME_COLS].sum(axis=1, numeric_only=True)

above_mod_bp = [c for c in BP_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_co = [c for c in CO_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_target_year["bp_affordable_total"] = (
    sd_apr_target_year["bp_units_total"] - sd_apr_target_year[above_mod_bp].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["co_affordable_total"] = (
    sd_apr_target_year["co_units_total"] - sd_apr_target_year[above_mod_co].sum(axis=1, numeric_only=True)
)

sd_apr_target_year["bp_affordable_share"] = sd_apr_target_year["bp_affordable_total"] / sd_apr_target_year["bp_units_total"]
sd_apr_target_year["co_affordable_share"] = sd_apr_target_year["co_affordable_total"] / sd_apr_target_year["co_units_total"]

production_by_year = sd_apr_target_year[[
    "jur_clean", YEAR_COL, "bp_units_total", "co_units_total",
    "bp_affordable_total", "co_affordable_total",
    "bp_affordable_share", "co_affordable_share",
]].rename(columns={YEAR_COL: "year"})
production_by_year.head()


## Load RHNA 6th Cycle targets

Separate CKAN package from APR (`rhna-progress-report`), but same portal.
This gives the RHNA *target* side -- the piece Table A2 doesn't have.

In [ ]:
RHNA_PACKAGE_URL = "https://data.ca.gov/api/3/action/package_show?id=rhna-progress-report"

rhna_package_json = get_json(RHNA_PACKAGE_URL)
rhna6_url, rhna6_name = find_resource_download_url(rhna_package_json, "6th Cycle RHNA Progress Report")
print("Using resource:", rhna6_name)
print("Download URL:", rhna6_url)

rhna_raw_path = RAW_DIR / "rhna6_progress_raw.csv"
if not rhna_raw_path.exists():
    resp = requests.get(rhna6_url, timeout=120)
    resp.raise_for_status()
    rhna_raw_path.write_bytes(resp.content)
    print("Saved to", rhna_raw_path)

rhna_raw = pd.read_csv(rhna_raw_path, low_memory=False)
print(rhna_raw.shape)
rhna_raw.head()


In [ ]:
# TODO: verify the actual jurisdiction column name once downloaded --
# print(rhna_raw.columns.tolist()) to check.
RHNA_JUR_COL = "Jurisdiction"  # <-- verify / update this

rhna_raw["jur_clean"] = rhna_raw[RHNA_JUR_COL].map(normalize_jurisdiction)
sd_rhna6 = rhna_raw[rhna_raw["jur_clean"].isin(sd_jur_keys)].copy()

print("San Diego rows in RHNA6 progress file:", len(sd_rhna6))
print("Jurisdictions found:", sorted(sd_rhna6["jur_clean"].unique()))
missing = sd_jur_keys - set(sd_rhna6["jur_clean"].unique())
if missing:
    print("Jurisdictions NOT found in RHNA6 file (check spelling/normalization):", sorted(missing))
sd_rhna6.head()


## Load City of San Diego building permits

Good news -- `data.sandiego.gov` turns out to run the same CKAN-style
portal as `data.ca.gov` (not Socrata, despite what many City of SD API
guides assume), so this uses the exact same `package_show` pattern as the
RHNA and APR sections above. No manual ID lookup needed.

The dataset has two resources -- **Active approvals** and **Closed
approvals** -- pull both and combine them, since a completed permit moves
from "active" to "closed" once it's finalized.


In [ ]:
PERMITS_PACKAGE_URL = "https://data.sandiego.gov/api/3/action/package_show?id=development-permits-set2"

permits_package_json = get_json(PERMITS_PACKAGE_URL)
active_url, active_name = find_resource_download_url(permits_package_json, "Active approvals")
closed_url, closed_name = find_resource_download_url(permits_package_json, "Closed approvals")
print("Active resource:", active_name, "->", active_url)
print("Closed resource:", closed_name, "->", closed_url)

permits_frames = []
for label, url in [("active", active_url), ("closed", closed_url)]:
    raw_path = RAW_DIR.parent / "sandiego_permits" / f"{label}_approvals_raw.csv"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    if not raw_path.exists():
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        raw_path.write_bytes(resp.content)
        print(f"Saved {label} approvals to", raw_path)
    df = pd.read_csv(raw_path, low_memory=False)
    df["approval_status"] = label
    permits_frames.append(df)

sd_permits_raw = pd.concat(permits_frames, ignore_index=True)
print(sd_permits_raw.shape)
sd_permits_raw.head()


## Load CA DOF population & housing estimates (E-5)

DOF does not publish a clean API for this -- the E-5 report is an Excel
file, and the filename/URL changes with each year's release, so this stays
a manual download.

**Confirmed structure** (from the 2020-2026 workbook): each year has two
sheets, e.g. `E5CountyState2025` (state/county totals) and
`E5CityCounty2025` (city-level detail, which is what we want). Within the
city-level sheet, header row is row 4, county names appear as their own
separator rows with no data, and each county's incorporated cities are
followed by an `Unincorporated` row (San Diego County's unincorporated
areas -- exactly the piece the RHNA/APR/permit data can't reach).

**Manual step required:** download the current E-5 workbook from
https://dof.ca.gov/forecasting/demographics/estimates/ and save it to
`data/raw/dof/e5_population_housing.xlsx` inside this workstream folder.


In [ ]:
DOF_RAW_PATH = RAW_DIR.parent / "dof" / "e5_population_housing.xlsx"
DOF_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)

DOF_SHEET_NAME = f"E5CityCounty{TARGET_YEAR}"

if not DOF_RAW_PATH.exists():
    print(
        f"DOF file not found at {DOF_RAW_PATH}. "
        "Download the current E-5 report from "
        "https://dof.ca.gov/forecasting/demographics/estimates/ and save it there."
    )
    dof_raw = pd.DataFrame()
else:
    dof_raw = pd.read_excel(DOF_RAW_PATH, sheet_name=DOF_SHEET_NAME, header=3)
    dof_raw = dof_raw.rename(columns={"County/City/State": "name"})
    dof_raw["name"] = dof_raw["name"].astype(str).str.strip()
    print(dof_raw.shape)
    dof_raw.head()


In [ ]:
# County-name rows have no population/housing data -- use that to forward-fill
# a "county" column onto every row that follows it, exactly like a merged-cell
# hierarchy would work in the original spreadsheet.
if not dof_raw.empty:
    dof_raw["is_county_header"] = dof_raw["Total"].isna() & dof_raw["name"].str.contains("County", na=False)
    dof_raw["county"] = dof_raw["name"].where(dof_raw["is_county_header"]).ffill()

    sd_dof = dof_raw[
        (dof_raw["county"] == "San Diego County")
        & (~dof_raw["is_county_header"])
        & (dof_raw["Total"].notna())
        & (~dof_raw["name"].isin(["Incorporated", "County Total"]))
    ].copy()

    # "Unincorporated" here represents the same entity as "county san diego"
    # elsewhere in this notebook and in the dashboard prototype.
    sd_dof["jur_clean"] = sd_dof["name"].map(
        lambda n: "county san diego" if n.strip() == "Unincorporated" else normalize_jurisdiction(n)
    )
    sd_dof["year"] = TARGET_YEAR

    print(f"DOF {TARGET_YEAR} rows for San Diego County jurisdictions:", len(sd_dof))
    print("Jurisdictions found:", sorted(sd_dof["jur_clean"].unique()))
    missing = sd_jur_keys - set(sd_dof["jur_clean"].unique())
    if missing:
        print("Jurisdictions NOT found in DOF file:", sorted(missing))
    display(sd_dof[["name", "county", "Total", "Household", "Group Quarters"]])
else:
    sd_dof = pd.DataFrame()


## Load Census ACS (2020-2024 5-year estimates)

Same pattern as Lauren's affordability notebook -- Census API, key entered
privately via `getpass` so it never gets saved into the notebook file.

In [ ]:
from getpass import getpass

CALIFORNIA_STATE_FIPS = "06"
SAN_DIEGO_COUNTY_FIPS = "073"

CENSUS_API_KEY = getpass("Paste your activated Census API key: ").strip()
if not CENSUS_API_KEY:
    raise ValueError("A Census API key is required.")

# Housing-production-relevant ACS variables: total population, total
# housing units, and tenure (owner/renter split) -- median income/rent are
# already covered in Lauren's affordability workstream, so not duplicated here.
ACS_VARS = {
    "B01003_001E": "population_total",
    "B25001_001E": "housing_units_total",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
}

acs_url = f"https://api.census.gov/data/{ACS_DATA_YEAR}/acs/acs5"
params = {
    "get": ",".join(["NAME", *ACS_VARS.keys()]),
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

acs_raw = get_json(acs_url, params=params)
acs_df = pd.DataFrame(acs_raw[1:], columns=acs_raw[0]).rename(columns=ACS_VARS)
acs_df["jur_clean"] = acs_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_acs = acs_df[acs_df["jur_clean"].isin(sd_jur_keys)].copy()
print(f"ACS {ACS_VINTAGE_LABEL} rows for San Diego County jurisdictions:", len(sd_acs))
sd_acs.head()


## Quick review: shape and completeness of everything loaded so far

In [ ]:
loaded = {
    "APR Table A2 (permits/completions)": sd_apr_target_year if "sd_apr_target_year" in dir() else pd.DataFrame(),
    "RHNA6 progress (targets)": sd_rhna6 if "sd_rhna6" in dir() else pd.DataFrame(),
    "City of SD permits": sd_permits_raw if "sd_permits_raw" in dir() else pd.DataFrame(),
    "DOF E-5 (population/housing)": sd_dof if "sd_dof" in dir() else pd.DataFrame(),
    "ACS 5-year": sd_acs if "sd_acs" in dir() else pd.DataFrame(),
}

for name, df in loaded.items():
    if df.empty:
        print(f"{name:35s} -> NOT LOADED YET")
    else:
        n_missing_cols = df.isna().sum()
        n_missing_cols = n_missing_cols[n_missing_cols > 0]
        print(f"{name:35s} -> {df.shape[0]} rows, {df.shape[1]} cols, "
              f"{len(n_missing_cols)} columns with missing values")


## Metric dictionary

In [ ]:
metric_dictionary = pd.DataFrame([
    {
        "output_metric": "bp_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of BP_*_INCOME columns",
        "definition": "Total housing units with a building permit issued, all income tiers, per jurisdiction-year",
    },
    {
        "output_metric": "co_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of CO_*_INCOME columns",
        "definition": "Total housing units with a certificate of occupancy (completed), all income tiers, per jurisdiction-year",
    },
    {
        "output_metric": "bp_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "bp_affordable_total / bp_units_total",
        "definition": "Share of permitted units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
    },
    {
        "output_metric": "co_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "co_affordable_total / co_units_total",
        "definition": "Share of completed units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
    },
])

metric_dictionary.to_csv(DOCS_DIR / "rhna_housing_production_metric_dictionary.csv", index=False)
metric_dictionary


## Export processed output

In [ ]:
output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv"
production_by_year.to_csv(output_path, index=False)
print("Saved:", output_path)


## Data Validation

Cross-check the freshly pulled HCD numbers against the values already baked
into the `housing-dashboard-prototype` repo's
`sd_apr_a2_city_year_supply.csv`. This is the core purpose of the
`chpd-dashboard-data-validation` repo -- confirming the dashboard's existing
numbers actually match the authoritative source.


In [ ]:
# Point this at your local clone of housing-dashboard-prototype.
# The two repos are separate, so update this path to wherever you cloned it.
BASELINE_PATH = Path(
    "../../housing-dashboard-prototype/data/processed/sd_apr_a2_city_year_supply.csv"
)

VALIDATION_JURISDICTIONS = ["san diego", "carlsbad", "chula vista", "poway"]

if BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH)
    baseline["jur_clean"] = baseline["jur_clean"].str.lower()

    comparison = production_by_year.merge(
        baseline,
        on="jur_clean",
        suffixes=("_hcd_fresh", "_dashboard_baseline"),
        how="inner",
    )

    failed_checks = comparison[
        comparison["bp_units_total_hcd_fresh"] != comparison["bp_units_total_dashboard_baseline"]
    ]

    print(f"Compared {len(comparison)} rows; {len(failed_checks)} mismatches found.")
    display(failed_checks[["jur_clean", "bp_units_total_hcd_fresh", "bp_units_total_dashboard_baseline"]])

    comparison.to_csv(PROCESSED_DIR / "rhna_housing_production_validation_report.csv", index=False)
else:
    print(
        f"Baseline file not found at {BASELINE_PATH}. "
        "Update BASELINE_PATH to point at your local housing-dashboard-prototype clone, "
        "or skip this validation step until both repos are cloned side by side."
    )


## Limitations and Next Steps

- **Year consistency:** this notebook targets 2025 for APR/RHNA,
  DOF, and City of SD permits, but ACS is pinned to its newest available
  vintage (2020-2024, i.e. 2024 data) since ACS structurally
  cannot produce 2025 data yet. Any output that joins ACS context
  onto 2025 production data is mixing two different data years by
  necessity -- label any such joined table clearly.
- HCD APR data is self-reported by jurisdictions; HCD does not independently
  verify it, and completeness/timing can vary by city and year. If the
  warning above about missing 2025 jurisdictions fired, note which
  cities hadn't filed yet as of when this notebook was run.
- This notebook currently pulls **Table A2 only** (permits/completions).
  RHNA 6th Cycle **targets** come from a separate HCD/SCAG allocation
  dataset and still need to be sourced and joined in -- see open item below.
- Column names in the raw HCD download have changed across publication
  years; the `# TODO` cells above should be re-checked any time this
  notebook is re-run against a newly refreshed file.
- Next: locate and pull the RHNA 6th Cycle target-allocation dataset,
  join it to `production_by_year` on `jur_clean`, and compute
  `pct_achieved = units_reported_total / rhna_target_total` to complete
  the RHNA-progress side of this workstream.
